# W7D4 — Evaluate Your RAG — Lab

**Week 7 · Day 4 · Retrieval, RAG and Recommenders** · Lab

Yesterday you built a system that answers. Today you find out whether any of the instructions you
wrote were actually followed — because none of them were guaranteed. They were asked for.

Three metrics, computed separately, because a single score tells you nothing about which component
to fix: **context recall** names the retriever, **faithfulness** names the generator, and **answer
relevance** names neither and catches the case where both worked and the answer was still useless.

Then the part that makes the numbers usable: you calibrate the judge that produces two of them
against your own hand labels, put V2 and V4 in one table so the refusal rate is read beside the
answerable scores, fix the five worst questions, and measure again.

**Time budget:** ~115 minutes. Sections 1–2 are the lab; Section 3 is a stretch you may finish at home.

<div dir="rtl" align="right">

# الأسبوع ٧ اليوم ٤ — قيّم نظام التوليد المعزّز بالاسترجاع

**الأسبوع السابع · اليوم الرابع · الاسترجاع والتوليد المعزّز والتوصية** · معمل

بالأمس بنيت نظامًا يجيب. واليوم تعرف هل اتُّبعت التعليمات التي كتبتها أصلًا — فلا شيء منها مضمون،
إنما طُلب طلبًا.

ثلاثة مقاييس تُحسب منفصلة، لأن الدرجة الواحدة لا تقول لك أي مكوّن تُصلح: **استدعاء السياق** يسمّي
المُسترجِع، و**الأمانة للمصدر** تسمّي المُولِّد، و**صلة الإجابة** لا تسمّي أيًّا منهما وتلتقط الحالة
التي عمل فيها الاثنان وبقيت الإجابة عديمة الفائدة.

ثم الجزء الذي يجعل الأرقام قابلة للاستعمال: تعاير الحَكَم الذي يُنتج اثنين منها بتسمياتك اليدوية،
وتضع V2 وV4 في جدول واحد ليُقرأ معدّل الرفض بجوار درجات الأسئلة القابلة للإجابة، وتُصلح أسوأ خمسة
أسئلة، وتقيس ثانيةً.

**الزمن المتوقّع:** نحو ١١٥ دقيقة. القسمان الأول والثاني هما المعمل، والقسم الثالث إضافي يمكن إكماله في المنزل.

</div>

> **This is your lab notebook.** Work through the hints — they tell you what to do and where
> to look, not what to type. Stuck for more than ten minutes on one task? Open the `_guided`
> version. That is not cheating; sitting stuck in silence is the only mistake. The full
> solution is released at the end of the day.

<div dir="rtl" align="right">

> **هذا دفتر المعمل الخاص بك.** اعمل وفق الإرشادات — فهي تخبرك بما يجب فعله وأين تبحث، لا بما
> تكتبه حرفيًا. إذا توقّفت أكثر من عشر دقائق عند مهمة واحدة فافتح نسخة `_guided`؛ هذا ليس غشًّا،
> والخطأ الوحيد هو أن تبقى متوقّفًا بصمت. ويُنشر الحل الكامل في نهاية اليوم.

</div>

## Learning objectives

By the end of this lab you can:

- Compute context recall, faithfulness and answer relevance separately, and say which component
  each one accuses.
- **Calibrate an LLM judge** against hand labels and report the agreement beside the score.
- Measure a refusal rate in a way that cannot be gamed by refusing everything.
- Use a diagnostic table to pick a fix, apply it, and measure whether it worked.
- Combine dense and keyword retrieval with reciprocal rank fusion and measure what it bought.
- Report latency and cost as first-class metrics and choose a configuration against a budget.

<div dir="rtl" align="right">

## أهداف التعلّم

في نهاية هذا المعمل تستطيع:

- أن تحسب استدعاء السياق والأمانة للمصدر وصلة الإجابة منفصلةً، وأن تقول أي مكوّن يتّهمه كلٌّ منها.
- أن **تعاير حَكَمًا لغويًّا** بتسميات يدوية وأن تذكر نسبة الاتفاق بجوار الدرجة.
- أن تقيس معدّل الرفض بطريقة لا يمكن خداعها برفض كل شيء.
- أن تستعمل جدول التشخيص لاختيار إصلاح، وأن تطبّقه، وأن تقيس هل نجح.
- أن تدمج الاسترجاع الكثيف والاسترجاع بالكلمات بدمج الرتب المتبادلة وأن تقيس ما اشتراه.
- أن تذكر الزمن والكلفة مقياسين من الدرجة الأولى وأن تختار ضبطًا في مواجهة ميزانية.

</div>

## About the data

**`rag_answers.parquet`** — yesterday's output: 80 rows, two prompt versions against 40 questions,
each row carrying the retrieved ids, the answer, the citations, the refusal flag, the latency and
the token count. `load_artefact` falls back to the reference copy if you did not finish Wednesday.

**`rag_eval_questions`** — the 40 questions with their gold sections. Thirty are answerable and
name the sections that answer them; ten are out of scope and name nothing.

**The ground truth is the expensive part, and it is worth saying out loud.** Those gold sections
were written by hand, before any system ran. That is what makes every number today a measurement
rather than an opinion, and it is the part a team skips when it says it does not have time to
evaluate. Forty questions took an afternoon.

**The judge is a language model, and it is not ground truth.** Two of today's three metrics come
from a model deciding whether a claim is supported. That model is wrong sometimes. Task 2 measures
how often, against ten answers you label yourself, and the notebook refuses to quote a faithfulness
number until that agreement is on record.

<div dir="rtl" align="right">

## عن البيانات

**`rag_answers.parquet`** — مُخرَج الأمس: ثمانون صفًّا، نسختا موجّه على أربعين سؤالًا، وفي كل صفّ
المعرّفات المسترجَعة والإجابة والاستشهادات وعلامة الرفض والزمن وعدد الرموز. ويرجع `load_artefact`
إلى النسخة المرجعية إن لم تُكمل الأربعاء.

**`rag_eval_questions`** — الأسئلة الأربعون بأقسامها المرجعية. ثلاثون قابلة للإجابة وتسمّي أقسامها،
وعشرة خارج النطاق ولا تسمّي شيئًا.

**والمرجع هو الجزء المكلف، ويستحقّ أن يُقال صراحةً.** فتلك الأقسام كُتبت يدويًّا قبل أن يعمل أي نظام.
وهذا ما يجعل كل رقم اليوم قياسًا لا رأيًا، وهو الجزء الذي يتخطّاه الفريق حين يقول إن لا وقت لديه
للتقييم. وأربعون سؤالًا استغرقت عصرًا واحدًا.

**والحَكَم نموذج لغوي، وليس مرجعًا.** فاثنان من مقاييس اليوم الثلاثة يأتيان من نموذج يقرّر هل
الادّعاء مُسنَد. وذلك النموذج يُخطئ أحيانًا. وتقيس المهمة الثانية كم يُخطئ، مقابل عشر إجابات تسمّيها
بنفسك، ولا يذكر الدفتر رقم أمانة حتى تُسجَّل تلك النسبة.

</div>

## Setup

Most of today runs with no model at all. Context recall, the refusal table, the re-chunking fix and
the hybrid retriever are arithmetic over yesterday's file and the corpus — no calls, no key, no
network. Only the judge in tasks 2 and 3 needs a model, and every judgement is cached like
yesterday's answers.

<div dir="rtl" align="right">

## الإعداد

معظم اليوم يعمل بلا نموذج البتّة. فاستدعاء السياق وجدول الرفض وإصلاح إعادة التقطيع والمُسترجِع
الهجين كلها حساب على ملف الأمس وعلى المُدوّنة — بلا نداءات ولا مفتاح ولا شبكة. ولا يحتاج نموذجًا إلا
الحَكَم في المهمّتين الثانية والثالثة، وكل حكم يُخزَّن كما خُزّنت إجابات الأمس.

</div>

In [ ]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, device, versions
from aiep.data import get_dataset, get_dataset_dir, load_artefact
from aiep.llm import chat, cache_stats, DEFAULT_MODEL
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, check_close, report
from aiep.viz import use_course_style, savefig

ensure("sentence-transformers", "rank-bm25", "openai", "python-dotenv", "matplotlib",
       "pandas", "pyarrow")
seed_everything(42)

import json
import re
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

use_course_style()

EMBEDDER = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K = 3
JUDGE_SAMPLE = 10          # answers you hand-label to calibrate the judge

CORPUS_DIR = get_dataset_dir("policy_docs")
MANIFEST = pd.read_csv(CORPUS_DIR / "manifest.csv").set_index("doc_id")
QUESTIONS = pd.read_parquet(get_dataset("rag_eval_questions")).set_index("qid")
ANSWERS = pd.read_parquet(load_artefact("rag_answers.parquet"))
CHUNKS = pd.read_parquet(load_artefact("chunks.parquet"))

V4 = ANSWERS[ANSWERS.version == "V4"].set_index("qid")
V2 = ANSWERS[ANSWERS.version == "V2"].set_index("qid")
ANSWERABLE = QUESTIONS[QUESTIONS.answerable].index.tolist()
OUT_OF_SCOPE = QUESTIONS[~QUESTIONS.answerable].index.tolist()

model = SentenceTransformer(EMBEDDER)

print(f"{len(ANSWERS)} answer rows · versions {sorted(ANSWERS.version.unique())}")
print(f"{len(ANSWERABLE)} answerable questions · {len(OUT_OF_SCOPE)} out of scope")
print(f"judge model: {DEFAULT_MODEL} · response cache: {cache_stats()}")
print(versions(), "| device:", device())

## Section 1 — Warm-up: three numbers by hand  (≈25 min)

This morning's worked example, hard-coded. One question, one answer, and a list.

```
Q:  can I return a downloaded game and how long does the refund take?

ground truth:  doc_7 §1   digital goods are excluded from the 14-day window
               doc_3 §2   refunds appear on the statement within five working days

retrieved:     doc_7 §1   ✓ the digital-goods rule
               doc_5 §2     store credit — related, not required
               doc_3 §1     "a refund is issued to the original payment method within 14 days"
```

The answer the system produced:

> A downloaded game cannot be refunded once the licence key is revealed. Refunds appear on your
> statement within five working days. The branch may offer store credit instead, and it does not
> expire.

Read it as a user: accurate, complete, well organised, answers both halves. **This is the answer you
would put in the demo.** Now compute the three numbers.

- **Context recall** — one of two required sources arrived: **0.50**. Look at the shape of the miss:
  retrieval returned `doc_3 §1`, the chunk sitting *next to* the one it needed. It looks like a hit.
- **Faithfulness** — three claims, two of them standing on retrieved text: **0.67**. Claim 2 is
  *true* — `doc_3 §2` says exactly that — and it was never retrieved, so the model produced it from
  somewhere else. True and ungrounded is the dangerous combination, because nothing in the output
  looks different.
- **Answer relevance** — it addressed both halves of the question: **1.00**.

Then read them together and write the sentence: **the answer was right and the system is broken.**

<div dir="rtl" align="right">

## القسم الأول — الإحماء: ثلاثة أرقام بيدك (نحو ٢٥ دقيقة)

مثال الصباح المشروح، مكتوبًا في الرمز. سؤال واحد وإجابة واحدة وقائمة.

والإجابة التي أنتجها النظام دقيقة وكاملة ومرتّبة وتجيب عن نصفَي السؤال. **وهي الإجابة التي تضعها في
العرض.** فاحسب الأرقام الثلاثة الآن.

- **استدعاء السياق** — وصل مصدر واحد من اثنين مطلوبين: **0.50**. وانظر شكل الفوات: أعاد الاسترجاع
  `doc_3 §1`، المقطع المجاور للذي احتاجه. ويبدو إصابةً وليس كذلك.
- **الأمانة للمصدر** — ثلاثة ادّعاءات، اثنان منها قائمان على نصّ مسترجَع: **0.67**. والادّعاء الثاني
  *صحيح* — إذ يقول `doc_3 §2` ذلك بعينه — ولم يُسترجَع قط، فأنتجه النموذج من مكان آخر. والصحيح غير
  المُسنَد هو التركيبة الخطرة، لأن لا شيء في المخرَج يبدو مختلفًا.
- **صلة الإجابة** — أجابت عن نصفَي السؤال: **1.00**.

ثم اقرأها معًا واكتب الجملة: **كانت الإجابة صحيحة والنظام معطوب.**

</div>

In [ ]:
WARM_GOLD = ["doc_7 §1", "doc_3 §2"]
WARM_RETRIEVED = ["doc_7 §1", "doc_5 §2", "doc_3 §1"]
WARM_CLAIMS = [
    ("a downloaded game cannot be refunded once the licence key is revealed", "doc_7 §1"),
    ("refunds appear on your statement within five working days", None),
    ("the branch may offer store credit instead, and it does not expire", "doc_5 §2"),
]
WARM_QUESTION_HALVES = ["can I return a downloaded game", "how long does the refund take"]
WARM_HALVES_ADDRESSED = [True, True]


def context_recall(retrieved, gold):
    """Of the sources the answer needed, how many did retrieval bring back?"""
    gold = [g for g in gold if g]
    if not gold:
        return None
    return len(set(retrieved) & set(gold)) / len(gold)


def faithfulness(claims):
    """Of the claims in the answer, how many stand on a retrieved chunk?"""
    if not claims:
        return None
    return sum(1 for _, support in claims if support is not None) / len(claims)


def answer_relevance(addressed):
    """Of the things the question asked for, how many did the answer address?"""
    return sum(addressed) / len(addressed)


WARM = {"recall": context_recall(WARM_RETRIEVED, WARM_GOLD),
        "faithfulness": faithfulness(WARM_CLAIMS),
        "relevance": answer_relevance(WARM_HALVES_ADDRESSED)}

print(f"context recall    {WARM['recall']:.2f}   "
      f"({len(set(WARM_RETRIEVED) & set(WARM_GOLD))} of {len(WARM_GOLD)} required sources)")
print(f"faithfulness      {WARM['faithfulness']:.2f}   "
      f"({sum(1 for _, s in WARM_CLAIMS if s)} of {len(WARM_CLAIMS)} claims grounded)")
print(f"answer relevance  {WARM['relevance']:.2f}   "
      f"({sum(WARM_HALVES_ADDRESSED)} of {len(WARM_HALVES_ADDRESSED)} halves addressed)")

missed = set(WARM_GOLD) - set(WARM_RETRIEVED)
neighbour = [r for r in WARM_RETRIEVED if r.split()[0] in {m.split()[0] for m in missed}]
print(f"\nthe source that never arrived: {sorted(missed)}")
print(f"what came instead: {neighbour} — the chunk next door, from the same document")
print("\nRelevance says the answer was on topic. Faithfulness says a third of it was not")
print("grounded. Recall says why. The three point at each other in the right order.")

**Your sentence.** Replace this text with one sentence that reads all three numbers together.

> …

<div dir="rtl" align="right">

**جملتك.** استبدل هذا النصّ بجملة واحدة تقرأ الأرقام الثلاثة معًا.

> …

</div>

## Section 2 — Core: six tasks  (≈60 min)

1. Context recall over the 30 answerable questions — mean, distribution, worst five.
2. Faithfulness with an LLM judge — **and calibrate it** against ten answers you label yourself.
3. Answer relevance over the same 30.
4. The refusal metric, and the V2-vs-V4 table. State which you would deploy.
5. Diagnose the five worst questions, apply the fix the diagnosis implies, and measure again.
6. Hybrid retrieval with reciprocal rank fusion, and the per-question deltas.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي: ست مهام (نحو ٦٠ دقيقة)

١. استدعاء السياق على الأسئلة الثلاثين — المتوسّط والتوزيع وأسوأ خمسة.
٢. الأمانة للمصدر بحَكَمٍ لغوي — **وعايره** بعشر إجابات تسمّيها بنفسك.
٣. صلة الإجابة على الثلاثين نفسها.
٤. مقياس الرفض، وجدول V2 مقابل V4. وقل أيّهما تنشر.
٥. شخّص أسوأ خمسة أسئلة، وطبّق الإصلاح الذي يقتضيه التشخيص، وقِس ثانيةً.
٦. الاسترجاع الهجين بدمج الرتب المتبادلة، والفروق لكل سؤال.

</div>

### Task 2.1 — context recall, over the whole set

Run `context_recall` over the 30 answerable questions using yesterday's V4 rows: the retrieved ids
are in the file, the gold sections are in the question set. Report the mean, the distribution, and
**list the worst five by name** — those five are task 5's input, and a mean with no worst-five list
beside it is a number you cannot act on.

This metric names the retriever and nothing else. If it is low, no prompt change and no better model
will help, because the text that answers the question was never in the input.

<div dir="rtl" align="right">

### المهمة ٢٫١ — استدعاء السياق على المجموعة كلها

شغّل `context_recall` على الأسئلة الثلاثين القابلة للإجابة من صفوف V4 للأمس: المعرّفات المسترجَعة في
الملف، والأقسام المرجعية في مجموعة الأسئلة. واذكر المتوسّط والتوزيع، و**اسرد أسوأ خمسة بأسمائها** —
فتلك الخمسة مُدخَل المهمة الخامسة، والمتوسّط بلا قائمة الأسوأ رقمٌ لا تستطيع التصرّف بناءً عليه.

وهذا المقياس يسمّي المُسترجِع ولا شيء غيره. فإن كان منخفضًا فلا تغيير موجّه ولا نموذج أفضل يفيد،
لأن النصّ الذي يجيب عن السؤال لم يكن في المُدخَل أصلًا.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Both columns are "|"-joined strings — split them before comparing, and drop empty
#    strings or an out-of-scope row will look like it had a gold section.
# 2) Reuse context_recall from the warm-up. One function, two places, no second version.
# 3) Sort the per-question results and take the five lowest. Print the question text too,
#    not just the id — task 5 is about reading them.
# Search: "pandas split string column apply set intersection"
# https://pandas.pydata.org/docs/reference/api/pandas.Series.str.split.html
#
# ١) العمودان سلسلتان موصولتان بـ"|" — قسّمهما قبل المقارنة، واحذف السلاسل الفارغة وإلا
#    بدا صفٌّ خارج النطاق وكأن له قسمًا مرجعيًّا.
# ٢) وأعِد استعمال `context_recall` من الإحماء. دالّة واحدة في موضعين بلا نسخة ثانية.
# ٣) ورتّب النتائج لكل سؤال وخذ الأدنى خمسة. واطبع نصّ السؤال أيضًا لا معرّفه فقط، فالمهمة
#    الخامسة عن قراءتها.
# ابحث عن: "pandas split string column apply set intersection"
# https://pandas.pydata.org/docs/reference/api/pandas.Series.str.split.html
# ────────────────────────────────────────────────────────────────────

def split_ids(value):
    """A "|"-joined id column, as a list, with the empties dropped."""
    return [part for part in str(value or "").split("|") if part]
# TODO: Compute context recall for every answerable question from the V4 rows, and collect qid, recall, retrieved and gold into one frame.
# مهمة: احسب استدعاء السياق لكل سؤال قابل للإجابة من صفوف V4، واجمع المعرّف والاستدعاء والمسترجَع والمرجعي في إطار واحد.
print(f"mean context recall@{TOP_K}: {RECALL.recall.mean():.3f} over "
      f"{len(RECALL)} answerable questions\n")
print("distribution:")
print(RECALL.recall.value_counts().sort_index().to_string())
print("\nthe five worst, and these are task 5's input:")
for qid in WORST_FIVE:
    print(f"  {qid} recall {RECALL.recall[qid]:.2f} · {QUESTIONS.question[qid]}")
    print(f"      gold      {RECALL.gold[qid]}")
    print(f"      retrieved {RECALL.retrieved[qid]}")

### Task 2.2 — faithfulness, and the judge you have to calibrate

Faithfulness asks: of the claims in the answer, how many stand on text that was actually retrieved?
Computing it needs the answer decomposed into claims and each claim checked against the context —
which is a language-model job, so you are now using a model to grade a model.

Build the judge: give it the retrieved context and the answer, ask it to list the claims and mark
each supported or not, and require JSON so the output can be parsed rather than read.

**Then calibrate it.** Take ten answers, read them yourself, and record your own verdict for each in
`HAND_LABELS`. Compare the judge against your labels and report the agreement. A judge you have not
calibrated is a number you cannot quote — and the sanity check at the bottom of this notebook will
not pass until those ten labels are filled in.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — الأمانة للمصدر، والحَكَم الذي يجب أن تعايره

تسأل الأمانة: من ادّعاءات الإجابة، كم واحدًا يقوم على نصّ استُرجع فعلًا؟ وحسابها يحتاج تفكيك الإجابة
إلى ادّعاءات وفحص كل ادّعاء مقابل السياق — وهذه وظيفة نموذج لغوي، فأنت الآن تستعمل نموذجًا ليصحّح
نموذجًا.

ابنِ الحَكَم: أعطه السياق المسترجَع والإجابة، واطلب منه سرد الادّعاءات ووسم كلٍّ مُسنَدًا أو لا،
واشترط JSON ليُحلَّل المخرَج لا ليُقرأ.

**ثم عايره.** خذ عشر إجابات، واقرأها بنفسك، وسجّل حكمك لكل واحدة في `HAND_LABELS`. وقارن الحَكَم
بتسمياتك واذكر نسبة الاتفاق. فالحَكَم غير المُعايَر رقمٌ لا تستطيع ذكره — ولن يجتاز فحص السلامة في
آخر هذا الدفتر حتى تُملأ تلك التسميات العشر.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) The judge prompt needs the context and the answer, and must ask for JSON:
#    {"claims": [{"claim": "...", "supported": true}]}. Say "reply with JSON only".
# 2) Models wrap JSON in code fences. Strip a leading ```json and a trailing ``` before
#    json.loads, and return None on a parse failure rather than crashing the loop.
# 3) The context you pass the judge must be the text of the chunks that were actually
#    retrieved for that question — not the whole corpus, or everything looks supported.
# Search: "llm as a judge faithfulness claim verification json output"
# https://arxiv.org/abs/2306.05685
#
# ١) يحتاج موجّه الحَكَم السياقَ والإجابة، ويجب أن يطلب JSON بالشكل
#    `{"claims": [{"claim": "...", "supported": true}]}`. وقل «أجب بـJSON فقط».
# ٢) والنماذج تغلّف JSON بأسوار رمز. فاحذف ```json في الأول و``` في الآخر قبل
#    `json.loads`، وأعِد `None` عند فشل التحليل بدل إسقاط الحلقة.
# ٣) والسياق الذي تمرّره للحَكَم يجب أن يكون نصّ المقاطع التي استُرجعت لذلك السؤال فعلًا —
#    لا المُدوّنة كلها، وإلا بدا كل شيء مُسنَدًا.
# ابحث عن: "llm as a judge faithfulness claim verification json output"
# https://arxiv.org/abs/2306.05685
# ────────────────────────────────────────────────────────────────────

CHUNK_TEXT = {}
for row in CHUNKS.itertuples():
    for section in split_ids(row.sections):
        CHUNK_TEXT.setdefault(section, row.text)
def context_for(qid):
    """The text of the chunks that were actually retrieved for this question."""
    return "\n\n".join(f"<doc id=\"{section}\">\n{CHUNK_TEXT.get(section, '')}\n</doc>"
                       for section in split_ids(V4.retrieved_ids[qid]))
def parse_json(text):
    """Models fence their JSON. Take the fence off, and fail softly."""
    cleaned = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", cleaned, re.DOTALL)
        try:
            return json.loads(match.group()) if match else None
        except json.JSONDecodeError:
            return None
# TODO: Write judge_faithfulness(question, answer, context): ask the model to decompose the answer into claims and mark each supported or not, and return the parsed claims.
# مهمة: اكتب `judge_faithfulness(question, answer, context)`: اطلب من النموذج تفكيك الإجابة إلى ادّعاءات ووسم كلٍّ مُسنَدًا أو لا، وأعِد الادّعاءات بعد التحليل.
# TODO: Score faithfulness for every answerable question from the judge's claim list.
# مهمة: احسب الأمانة لكل سؤال قابل للإجابة من قائمة ادّعاءات الحَكَم.
print(f"mean faithfulness: {FAITH.faithfulness.mean():.3f} over "
      f"{int(FAITH.faithfulness.notna().sum())} scored answers "
      f"({int(FAITH.faithfulness.isna().sum())} unparseable)")
print(f"answers with at least one ungrounded claim: "
      f"{int((FAITH.faithfulness < 1.0).sum())}\n")
for row in FAITH[FAITH.faithfulness < 1.0].head(3).itertuples():
    print(f"  {row.Index}: {row.faithfulness:.2f} — ungrounded: {row.unsupported[:100]}")

#### Calibrating the judge

Ten answers. Read each one against its retrieved context and decide, yourself, whether **every**
claim in it is supported. Put `True` or `False` in `HAND_LABELS` — `True` meaning fully grounded.

Then compare. The agreement rate goes in the report beside every faithfulness number you quote, and
the disagreements are worth reading one by one: a judge that is systematically generous is a
different problem from one that is noisy.

<div dir="rtl" align="right">

#### معايرة الحَكَم

عشر إجابات. اقرأ كلًّا منها مقابل سياقها المسترجَع وقرّر بنفسك هل **كل** ادّعاء فيها مُسنَد. وضع
`True` أو `False` في `HAND_LABELS`، و`True` تعني مُسنَدًا بالكامل.

ثم قارن. وتذهب نسبة الاتفاق في التقرير بجوار كل رقم أمانة تذكره، وتستحقّ مواضع الاختلاف قراءةً واحدًا
واحدًا: فالحَكَم المتساهل بانتظام مشكلةٌ غير الحَكَم المتذبذب.

</div>

In [ ]:
# Print the ten answers to label. Read each against its context before you decide.
CALIBRATION_QIDS = ANSWERABLE[:JUDGE_SAMPLE]

for qid in CALIBRATION_QIDS:
    print(f"=== {qid} · {QUESTIONS.question[qid]}")
    print(f"retrieved: {V4.retrieved_ids[qid]}")
    print(textwrap.fill(f"answer: {V4.answer[qid]}", 96))
    print(f"judge said: {FAITH.faithfulness[qid]}  "
          f"(ungrounded: {FAITH.unsupported[qid] or 'none'})\n")

In [ ]:
# Your labels. True = every claim in the answer is supported by the retrieved context.
# Replace these with your own reading — they are the only ground truth in this notebook
# that you produced yourself, and the judge is scored against them, not the other way round.
HAND_LABELS = {}

judged = {qid: (FAITH.faithfulness[qid] == 1.0) for qid in CALIBRATION_QIDS
          if FAITH.faithfulness[qid] is not None}
overlap = [qid for qid in HAND_LABELS if qid in judged]
AGREEMENT = (float(np.mean([HAND_LABELS[qid] == judged[qid] for qid in overlap]))
             if overlap else None)

print(f"hand-labelled: {len(HAND_LABELS)} of {JUDGE_SAMPLE} required")
if AGREEMENT is None:
    print("No labels yet — fill HAND_LABELS in the cell above before quoting a faithfulness "
          "number.\nلا تسميات بعد — املأ HAND_LABELS قبل ذكر أي رقم أمانة.")
else:
    print(f"judge agreement with your labels: {AGREEMENT:.0%} on {len(overlap)} answers")
    for qid in overlap:
        if HAND_LABELS[qid] != judged[qid]:
            print(f"  disagreement on {qid}: you said {HAND_LABELS[qid]}, "
                  f"judge said {judged[qid]}")

### Task 2.3 — answer relevance

Relevance asks a different question from both of the others: did the answer address what was asked?
An answer can be perfectly grounded in the retrieved context and still not answer the question — it
happens most often when retrieval returns a related chunk and the model writes a competent summary
of the wrong thing.

Score it with the judge, on a 0–1 scale, over the same 30. Then look at the questions where
relevance is high and faithfulness is low, and the ones where it is the other way round. Those two
groups have different causes and different fixes, which is the entire reason these are three
numbers instead of one.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — صلة الإجابة

تسأل الصلة سؤالًا غير سؤال الاثنين الآخرين: هل أجابت الإجابة عمّا سُئل؟ فقد تكون الإجابة مُسنَدةً
تمامًا إلى السياق المسترجَع ولا تجيب عن السؤال — ويحدث هذا غالبًا حين يعيد الاسترجاع مقطعًا ذا صلة
فيكتب النموذج تلخيصًا متقنًا للشيء الخطأ.

قِسها بالحَكَم على مقياس من صفر إلى واحد، على الثلاثين نفسها. ثم انظر إلى الأسئلة التي ارتفعت فيها
الصلة وانخفضت الأمانة، وإلى العكس. فللمجموعتين سببان مختلفان وعلاجان مختلفان، وهذا هو سبب كونها
ثلاثة أرقام لا رقمًا واحدًا.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) The relevance judge does not need the documents — only the question and the answer.
#    Sending the context invites it to grade groundedness again, which you already have.
# 2) Ask for {"relevance": 0.0-1.0, "reason": "..."} and parse it the same way.
# 3) Join relevance to faithfulness and recall in one frame, then look at the rows where
#    the three disagree — that is where the diagnosis lives.
# Search: "answer relevance rag evaluation llm judge"
# https://arxiv.org/abs/2309.15217
#
# ١) لا يحتاج حَكَم الصلة الوثائقَ — بل السؤال والإجابة فقط. فإرسال السياق يدعوه إلى
#    إعادة تقييم الإسناد، وهو عندك أصلًا.
# ٢) واطلب `{"relevance": 0.0-1.0, "reason": "..."}` وحلّله بالطريقة نفسها.
# ٣) واضمم الصلة إلى الأمانة والاستدعاء في إطار واحد، ثم انظر إلى الصفوف التي تختلف فيها
#    الثلاثة — فهناك يسكن التشخيص.
# ابحث عن: "answer relevance rag evaluation llm judge"
# https://arxiv.org/abs/2309.15217
# ────────────────────────────────────────────────────────────────────

# TODO: Write judge_relevance(question, answer) returning a 0-1 score, and score all 30.
# مهمة: اكتب `judge_relevance(question, answer)` تُعيد درجة بين ٠ و١، وقِس الثلاثين كلها.
METRICS = pd.concat([RECALL.recall, FAITH.faithfulness, RELEVANCE], axis=1)
print(METRICS.describe().round(3).to_string())
disagree = METRICS[(METRICS.relevance >= 0.8) & (METRICS.faithfulness < 1.0)]
print(f"\nrelevant but not fully grounded: {len(disagree)} questions "
      f"{disagree.index.tolist()}")
print("Those are the dangerous ones: a user reads them as good answers.")

### Task 2.4 — the refusal metric, and why it needs the other half beside it

Ten questions in this set are out of scope. A correct refusal is a *correct answer* for those ten,
and yesterday's two prompt versions behave differently on them: V2 answers, V4 declines.

Build the table with **both halves in it**:

| system | correct refusals (of 10) | mean recall on the 30 | mean faithfulness on the 30 |
|---|---|---|---|

The second and third columns are what stop the first from being gamed. A system that refuses every
question scores 10/10 on refusals and is useless, and a refusal rate quoted on its own cannot tell
you which system you are looking at. Then state, in one sentence, which you would deploy.

<div dir="rtl" align="right">

### المهمة ٢٫٤ — مقياس الرفض، ولماذا يحتاج نصفه الآخر بجواره

عشرة أسئلة في هذه المجموعة خارج النطاق. والرفض الصحيح *إجابة صحيحة* لتلك العشرة، ونسختا موجّه الأمس
تختلفان فيها: يجيب V2 ويرفض V4.

ابنِ الجدول **بنصفيه**: الرفض الصحيح من عشرة، ومتوسّط الاستدعاء على الثلاثين، ومتوسّط الأمانة عليها.

والعمودان الثاني والثالث هما ما يمنع خداع الأول. فالنظام الذي يرفض كل سؤال يسجّل عشرة من عشرة في
الرفض وهو عديم الفائدة، ومعدّل الرفض وحده لا يقول لك أي نظام تنظر إليه. ثم قل في جملة واحدة أيّهما
تنشر.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) The refused flag is already in yesterday's file for both versions. A correct refusal
#    is refused == True on an out-of-scope question.
# 2) For the answerable half, V2 and V4 retrieved the same chunks — so their recall is
#    identical by construction, and saying that is part of the answer.
# 3) Score V2's faithfulness on a handful of the answerable questions rather than all 30
#    if you are short of calls, and say how many you scored.
# Search: "selective prediction abstention rate evaluation"
# https://arxiv.org/abs/2309.15217
#
# ١) علامة الرفض في ملف الأمس للنسختين. والرفض الصحيح هو `refused == True` على سؤال خارج
#    النطاق.
# ٢) وفي النصف القابل للإجابة استرجع V2 وV4 المقاطع نفسها، فاستدعاؤهما متطابق بحكم
#    البناء، وقول ذلك جزء من الجواب.
# ٣) وقِس أمانة V2 على بضعة أسئلة قابلة للإجابة بدل الثلاثين إن ضاقت النداءات، وقل كم
#    قِست.
# ابحث عن: "selective prediction abstention rate evaluation"
# https://arxiv.org/abs/2309.15217
# ────────────────────────────────────────────────────────────────────

# TODO: Build the two-system table: correct refusals out of 10, mean recall on the 30, and mean faithfulness on the answerable questions.
# مهمة: ابنِ جدول النظامين: الرفض الصحيح من عشرة، ومتوسّط الاستدعاء على الثلاثين، و متوسّط الأمانة على الأسئلة القابلة للإجابة.
print(REFUSAL_TABLE.round(3).to_string(index=False))
print("\nBoth systems retrieved the same chunks, so their context recall is identical by")
print("construction. The refusal column is the only thing that moved, and it moved because")
print("of one sentence in the system message.")

**Which do you deploy?** One sentence, and name the number you are deciding on. Replace this text.

> …

<div dir="rtl" align="right">

**أيّهما تنشر؟** جملة واحدة، وسمِّ الرقم الذي تقرّر بناءً عليه. استبدل هذا النصّ.

> …

</div>

### Task 2.5 — diagnose, fix, and measure again

Take the five worst questions from task 1 and read the diagnostic table from this morning:

| symptom | cause | fix |
|---|---|---|
| The right document came back, the wrong section | chunks too large, or a boundary cut the answer in half | re-chunk |
| Nothing from the right document came back | vocabulary mismatch between question and text | hybrid retrieval, task 6 |
| The answer needs two documents and one arrived | `top_k` too small, or the two share no vocabulary | raise k, or accept it |

The usual cause is chunking, so apply that fix: re-chunk the corpus at a different size, re-embed,
re-retrieve **for those five questions only**, and compare recall before and after.

**Measure the whole set afterwards, not only the five.** A fix that repairs five questions and
breaks three others is not a fix, and you cannot see that from the five. This is the single most
common way an evaluation gets used to prove something false.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — شخّص، وأصلح، وقِس ثانيةً

خذ أسوأ خمسة أسئلة من المهمة الأولى واقرأ جدول التشخيص من هذا الصباح: فإن عادت الوثيقة الصحيحة
والقسم الخطأ فالمقاطع كبيرة أو الحدّ شقّ الإجابة، والعلاج إعادة التقطيع؛ وإن لم يعد شيء من الوثيقة
الصحيحة فالمفردات غير متطابقة، والعلاج الاسترجاع الهجين في المهمة السادسة؛ وإن احتاجت الإجابة
وثيقتين ووصلت واحدة فـ`top_k` صغير أو الوثيقتان لا تشتركان في مفردة.

والسبب المعتاد هو التقطيع، فطبّق ذلك العلاج: أعِد تقطيع المُدوّنة بحجم مختلف، وأعِد التضمين، وأعِد
الاسترجاع **لتلك الخمسة وحدها**، وقارن الاستدعاء قبل وبعد.

**ثم قِس المجموعة كلها بعدها، لا الخمسة فقط.** فالإصلاح الذي يُصلح خمسة ويكسر ثلاثة ليس إصلاحًا، ولا
تراه من الخمسة. وهذه أشيع طريقة يُستعمل بها التقييم لإثبات شيء غير صحيح.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Re-chunk from the corpus: read each document, cut it at a different size, and record
#    which sections each chunk covers — the same span logic as Monday.
# 2) Embed the new chunks, embed the questions, and take the top TOP_K by cosine. No
#    model call is needed here: recall is about retrieval only.
# 3) Report three numbers: recall on the five before, recall on the five after, and mean
#    recall over all 30 after. The third is the one that decides.
# Search: "chunk size retrieval recall re-index evaluation"
# https://www.sbert.net/examples/applications/semantic-search/README.html
#
# ١) أعِد التقطيع من المُدوّنة: اقرأ كل وثيقة، واقطعها بحجم مختلف، وسجّل الأقسام التي
#    يغطّيها كل مقطع — بمنطق المجالات نفسه الذي في يوم الاثنين.
# ٢) وضمّن المقاطع الجديدة، وضمّن الأسئلة، وخذ أفضل `TOP_K` بجيب التمام. ولا حاجة لنداء
#    نموذج هنا: فالاستدعاء عن الاسترجاع وحده.
# ٣) واذكر ثلاثة أرقام: استدعاء الخمسة قبل، واستدعاءها بعد، ومتوسّط الثلاثين بعد. والثالث
#    هو الذي يحسم.
# ابحث عن: "chunk size retrieval recall re-index evaluation"
# https://www.sbert.net/examples/applications/semantic-search/README.html
# ────────────────────────────────────────────────────────────────────

NEW_SIZE, NEW_OVERLAP = 250, 25       # a smaller chunk than Monday's winner
def read_document(doc_id):
    """Document text plus (start, end, section number) spans — Monday's function."""
    lines = (CORPUS_DIR / "docs" / f"{doc_id}.md").read_text(encoding="utf-8").splitlines()
    sections, current = [], None
    for line in lines:
        if line.startswith("## "):
            current = [line[3:].strip().split(". ", 1)[-1], []]
            sections.append(current)
        elif line.startswith("#") or line.startswith("*") or not line.strip():
            continue
        elif current is not None:
            current[1].append(line.strip())
    text, spans, at = "", [], 0
    for number, (heading, body) in enumerate(sections, start=1):
        part = f"{heading}. {' '.join(body)} "
        spans.append((at, at + len(part), number))
        text += part
        at += len(part)
    return text, spans
# TODO: Re-chunk the corpus at NEW_SIZE, embed it, and retrieve for every answerable question.
# مهمة: أعِد تقطيع المُدوّنة عند `NEW_SIZE`، وضمّنها، واسترجع لكل سؤال قابل للإجابة.
BEFORE_FIVE = RECALL.recall[WORST_FIVE].mean()
AFTER_FIVE = AFTER[WORST_FIVE].mean()
IMPROVED = int((AFTER[WORST_FIVE] > RECALL.recall[WORST_FIVE]).sum())
comparison = pd.DataFrame({"before": RECALL.recall, "after": AFTER})
comparison["delta"] = comparison.after - comparison.before
print(f"re-chunked at {NEW_SIZE} characters with {NEW_OVERLAP} overlap → "
      f"{len(RECHUNKED)} chunks (was {len(CHUNKS[CHUNKS.strategy == CHUNKS.strategy.iloc[0]])})\n")
print(f"the five targeted: {BEFORE_FIVE:.3f} → {AFTER_FIVE:.3f} "
      f"({IMPROVED} of 5 improved)")
print(f"the whole set:     {RECALL.recall.mean():.3f} → {AFTER.mean():.3f}\n")
print("questions the fix broke:")
broken = comparison[comparison.delta < 0]
print(broken.round(2).to_string() if len(broken) else "  none")

### Task 2.6 — hybrid retrieval, and where it pays

Tuesday's task 6 showed dense retrieval losing to BM25 on queries that hang on a literal identifier.
Hybrid retrieval is the obvious response: run both, then merge the two rankings.

**Reciprocal rank fusion** is provided below and it is three lines. Each document's score is the sum
over retrievers of `1 / (k + rank)`, with `k = 60` by convention. It needs no score normalisation,
which is the reason it is used in practice — dense distances and BM25 scores are not on comparable
scales and any attempt to put them on one is a tuning parameter you will have to defend.

Recompute context recall across all 30 with the fused ranking and record the per-question deltas.
Then check where the gain is concentrated: it should be largest on the identifier-style questions,
which is the prediction Tuesday's table made.

<div dir="rtl" align="right">

### المهمة ٢٫٦ — الاسترجاع الهجين، وأين يؤتي ثمرته

أظهرت مهمة الثلاثاء السادسة خسارة الاسترجاع الكثيف أمام BM25 في الاستعلامات المعلَّقة بمعرّف حرفي.
والاسترجاع الهجين هو الردّ البديهي: شغّل الاثنين ثم ادمج الترتيبين.

و**دمج الرتب المتبادلة** مُعطى أدناه وهو ثلاثة أسطر. فدرجة كل وثيقة مجموع `1 / (k + rank)` على
المُسترجِعَين، و`k = 60` اصطلاحًا. ولا يحتاج تطبيع درجات، ولهذا يُستعمل عمليًّا — فمسافات الكثيف
ودرجات BM25 ليست على مقياس واحد، وكل محاولة لوضعها على مقياس واحد معامل ضبط ستضطرّ للدفاع عنه.

أعِد حساب استدعاء السياق على الثلاثين بالترتيب المدموج وسجّل الفروق لكل سؤال. ثم تحقّق أين تتركّز
الزيادة: ينبغي أن تكون أكبر في الأسئلة ذات المعرّفات، وهو ما تنبّأ به جدول الثلاثاء.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Build BM25 over the same chunk texts the dense retriever uses, with the same
#    tokenisation on both sides — lower-case and strip the punctuation off identifiers.
# 2) rank lists, not score lists: for each retriever, argsort and take the order. RRF
#    only ever looks at positions.
# 3) Group the deltas by the question's kind column. The identifier questions are where
#    the argument for hybrid lives, and one mean over all 30 hides it.
# Search: "reciprocal rank fusion hybrid retrieval bm25 dense"
# https://learn.microsoft.com/en-us/azure/search/hybrid-search-ranking
#
# ١) ابنِ BM25 على نصوص المقاطع نفسها التي يستعملها الكثيف، بالتقطيع نفسه في الجانبين —
#    حروف صغيرة ونزع الترقيم عن المعرّفات.
# ٢) وقوائم رتب لا قوائم درجات: لكل مُسترجِع رتّب بـ`argsort` وخذ الترتيب. فدمج الرتب لا
#    ينظر إلا إلى المواضع.
# ٣) وجمّع الفروق حسب عمود `kind` للسؤال. فأسئلة المعرّفات هي موضع حجّة الهجين، والمتوسّط
#    الواحد على الثلاثين يخفيها.
# ابحث عن: "reciprocal rank fusion hybrid retrieval bm25 dense"
# https://learn.microsoft.com/en-us/azure/search/hybrid-search-ranking
# ────────────────────────────────────────────────────────────────────

RRF_K = 60
def reciprocal_rank_fusion(rankings, k=RRF_K):
    """Merge ranked id lists by position. No score normalisation, by design."""
    fused = {}
    for ranking in rankings:
        for position, item in enumerate(ranking):
            fused[item] = fused.get(item, 0.0) + 1.0 / (k + position + 1)
    return sorted(fused, key=fused.get, reverse=True)
def tokenise(text):
    return [word.lower().strip(".,()§#") for word in text.split()]
# TODO: Build a BM25 index over the re-chunked corpus, fuse it with the dense ranking, and recompute context recall for every answerable question.
# مهمة: ابنِ فهرس BM25 على المُدوّنة المُعاد تقطيعها، وادمجه مع الترتيب الكثيف، و أعِد حساب استدعاء السياق لكل سؤال قابل للإجابة.
DELTAS = pd.DataFrame({"dense": AFTER, "hybrid": HYBRID,
                       "kind": [QUESTIONS.kind[qid] for qid in ANSWERABLE]},
                      index=ANSWERABLE)
DELTAS["delta"] = DELTAS.hybrid - DELTAS.dense
print(f"mean context recall — dense {DELTAS.dense.mean():.3f}, "
      f"hybrid {DELTAS.hybrid.mean():.3f}\n")
print(DELTAS.groupby("kind")[["dense", "hybrid", "delta"]].mean().round(3).to_string())
print(f"\nquestions helped: {int((DELTAS.delta > 0).sum())} · "
      f"unchanged: {int((DELTAS.delta == 0).sum())} · "
      f"hurt: {int((DELTAS.delta < 0).sum())}")

## Section 3 — Stretch: latency and cost are metrics too  (≈30 min)

Three configurations, one table: yesterday's V4 pipeline, the hybrid retriever, and `top_k = 10`.
For each, record **p50 and p95 latency** and the **total tokens**.

Then answer the question a product owner actually asks: *given a three-second budget per query,
which configuration ships?* A configuration whose median is comfortable and whose p95 is over budget
is a configuration that fails for one user in twenty, every day, and the p50 is the number that will
be in the slide deck.

Note which of your latency numbers came from a cache hit — those are not latency measurements at
all, and the `cached` column is in yesterday's file precisely so you can exclude them.

<div dir="rtl" align="right">

## القسم الثالث — التوسّع: الزمن والكلفة مقياسان أيضًا (نحو ٣٠ دقيقة)

ثلاثة ضبوط في جدول واحد: مسار V4 من الأمس، والمُسترجِع الهجين، و`top_k = 10`. وسجّل لكلٍّ **الزمن
عند المئين الخمسين والخامس والتسعين** و**مجموع الرموز**.

ثم أجب عن السؤال الذي يسأله مالك المنتج فعلًا: *بميزانية ثلاث ثوانٍ للاستعلام، أي ضبط يُنشر؟*
فالضبط الذي وسيطه مريح ومئينه الخامس والتسعون فوق الميزانية ضبطٌ يُخفق لمستخدم من كل عشرين، كل يوم،
والوسيط هو الرقم الذي سيكون في العرض.

ولاحظ أي أرقام زمنك جاءت من المخزن المؤقّت — فتلك ليست قياسات زمن أصلًا، وعمود `cached` في ملف
الأمس موجود لهذا بالضبط: لتستبعدها.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Yesterday's file has latency_s, total_tokens and cached per row. Filter out the
#    cached rows before computing any latency statistic, and say how many were left.
# 2) p50 and p95 are Series.quantile(0.5) and .quantile(0.95). Report both, always —
#    a median alone hides the failure mode.
# 3) If every row is cached (you ran this from the reference cache), say so and report
#    the token counts only. An honest missing number beats an invented one.
# Search: "p50 p95 latency percentile pandas quantile"
# https://pandas.pydata.org/docs/reference/api/pandas.Series.quantile.html
#
# ١) في ملف الأمس `latency_s` و`total_tokens` و`cached` لكل صفّ. فاستبعد الصفوف المخزَّنة
#    قبل حساب أي إحصاء زمني، وقل كم بقي.
# ٢) والمئينان هما `Series.quantile(0.5)` و`.quantile(0.95)`. واذكرهما دائمًا — فالوسيط
#    وحده يخفي نمط الإخفاق.
# ٣) وإن كانت كل الصفوف مخزَّنة (شغّلت من المخزن المرجعي) فقل ذلك واذكر الرموز فقط.
#    فالرقم الغائب بأمانة خير من رقم مخترع.
# ابحث عن: "p50 p95 latency percentile pandas quantile"
# https://pandas.pydata.org/docs/reference/api/pandas.Series.quantile.html
# ────────────────────────────────────────────────────────────────────

BUDGET_SECONDS = 3.0
# TODO: Build the three-way latency and token table from yesterday's rows, excluding cache hits from every latency statistic.
# مهمة: ابنِ جدول الزمن والرموز الثلاثي من صفوف الأمس، مستبعدًا قراءات المخزن من كل إحصاء زمني.
print(COST.round(3).to_string(index=False))
if COST.live_calls.sum() == 0:
    print("\nEvery row came from the response cache, so there is no latency to report here.")
    print("Re-run with a key to measure it, and do not quote a cached number as a latency.")
else:
    print(f"\nbudget: {BUDGET_SECONDS}s per query at p95")

**Which configuration ships?** One sentence, naming the metric and the budget. Replace this text.

> …

<div dir="rtl" align="right">

**أي ضبط يُنشر؟** جملة واحدة تسمّي المقياس والميزانية. استبدل هذا النصّ.

> …

</div>

## Save the artefacts

`rag_eval.parquet` — one row per question with every metric on it: recall, faithfulness, relevance,
refused, correct refusal, latency, tokens.

`eval_report.md` — the thing a reviewer reads instead of the notebook. The three hand-computed
warm-up numbers, the metric distributions, **the judge calibration**, the V2-vs-V4 refusal table,
the before-and-after of the fix, and the hybrid deltas. Write it as the evaluation section of a
report, because that is what it becomes: the capstone is graded on exactly this shape.

<div dir="rtl" align="right">

## احفظ المُخرَجات

`rag_eval.parquet` — صفّ لكل سؤال بكل مقاييسه: الاستدعاء والأمانة والصلة والرفض والرفض الصحيح
والزمن والرموز.

و`eval_report.md` — ما يقرؤه المراجع بدل الدفتر. فيه أرقام الإحماء الثلاثة المحسوبة يدويًّا، وتوزيعات
المقاييس، و**معايرة الحَكَم**، وجدول الرفض بين V2 وV4، وقبل الإصلاح وبعده، وفروق الهجين. اكتبه كقسم
تقييم في تقرير، لأنه سيصير ذلك: ومشروع التخرّج مُقيَّم على هذا الشكل بعينه.

</div>

In [ ]:
ARTEFACT_DIR.mkdir(parents=True, exist_ok=True)

EVAL = pd.DataFrame(index=QUESTIONS.index)
EVAL["kind"] = QUESTIONS.kind
EVAL["answerable"] = QUESTIONS.answerable
EVAL["recall"] = RECALL.recall
EVAL["recall_after_fix"] = AFTER
EVAL["recall_hybrid"] = HYBRID
EVAL["faithfulness"] = FAITH.faithfulness
EVAL["relevance"] = RELEVANCE
EVAL["refused_v4"] = V4.refused
EVAL["refused_v2"] = V2.refused
EVAL["correct_refusal"] = (~QUESTIONS.answerable) & V4.refused
EVAL["latency_s"] = V4.latency_s
EVAL["total_tokens"] = V4.total_tokens
EVAL.to_parquet(ARTEFACT_DIR / "rag_eval.parquet")

report_lines = [
    "# RAG evaluation — Olo Retail policy assistant",
    "",
    f"Forty questions, thirty answerable with hand-written gold sections, ten out of scope.",
    f"Judge: `{DEFAULT_MODEL}`.",
    "",
    "## Warm-up, computed by hand",
    "",
    f"| metric | value |", "|---|---|",
    f"| context recall | {WARM['recall']:.2f} |",
    f"| faithfulness | {WARM['faithfulness']:.2f} |",
    f"| answer relevance | {WARM['relevance']:.2f} |",
    "",
    "The answer was right and the system was broken: a required source never arrived, and the",
    "model filled the gap with a claim that is true and ungrounded.",
    "",
    "## Judge calibration",
    "",
    (f"Agreement with hand labels: **{AGREEMENT:.0%}** on {len(overlap)} answers."
     if AGREEMENT is not None else
     "**Not calibrated.** No hand labels were recorded, so no faithfulness number here is quotable."),
    "",
    "## Metrics over the thirty answerable questions",
    "",
    METRICS.describe().round(3).to_markdown(),
    "",
    "## Refusal, both systems",
    "",
    REFUSAL_TABLE.round(3).to_markdown(index=False),
    "",
    "## The chunking fix",
    "",
    f"- targeted five: {BEFORE_FIVE:.3f} → {AFTER_FIVE:.3f} ({IMPROVED} of 5 improved)",
    f"- whole set: {RECALL.recall.mean():.3f} → {AFTER.mean():.3f}",
    "",
    "## Hybrid retrieval",
    "",
    DELTAS.groupby("kind")[["dense", "hybrid", "delta"]].mean().round(3).to_markdown(),
    "",
    "## Latency and cost",
    "",
    COST.round(3).to_markdown(index=False),
    "",
]
(ARTEFACT_DIR / "eval_report.md").write_text("\n".join(report_lines), encoding="utf-8")

print(f"rag_eval.parquet — {len(EVAL)} rows, {EVAL.shape[1]} columns")
print(f"eval_report.md   — {len(report_lines)} lines at {ARTEFACT_DIR / 'eval_report.md'}")
print(EVAL[["recall", "faithfulness", "relevance", "correct_refusal"]].describe().round(3).to_string())

## Sanity check

<div dir="rtl" align="right">

## فحص سلامة

</div>

In [ ]:
check(np.isclose(WARM["recall"], 0.50) and np.isclose(WARM["faithfulness"], 2 / 3, atol=0.005)
      and np.isclose(WARM["relevance"], 1.00),
      f"the warm-up must reproduce the slide's three numbers — got recall {WARM['recall']:.2f}, "
      f"faithfulness {WARM['faithfulness']:.2f}, relevance {WARM['relevance']:.2f}, expected "
      f"0.50, 0.67, 1.00",
      f"يجب أن يُعيد الإحماء أرقام الشريحة الثلاثة — والناتج: الاستدعاء {WARM['recall']:.2f}، "
      f"والأمانة {WARM['faithfulness']:.2f}، والصلة {WARM['relevance']:.2f}، والمتوقّع 0.50 و0.67 و1.00")

check(len(HAND_LABELS) >= JUDGE_SAMPLE and AGREEMENT is not None,
      f"the judge must be calibrated on at least {JUDGE_SAMPLE} hand-labelled answers before any "
      f"faithfulness number is quotable — you labelled {len(HAND_LABELS)}. This check failing is "
      f"the notebook refusing to let you report a number you have not checked",
      f"يجب معايرة الحَكَم على {JUDGE_SAMPLE} إجابات مُسمّاة يدويًّا على الأقل قبل أن يكون أي رقم أمانة "
      f"قابلًا للذكر — وقد سمّيت {len(HAND_LABELS)}. وفشل هذا الفحص هو رفض الدفتر أن يدعك تذكر رقمًا "
      f"لم تتحقّق منه")

check(len(REFUSAL_TABLE) == 2 and REFUSAL_TABLE.correct_refusals.notna().all(),
      f"the refusal table must contain both systems with their correct-refusal counts — got "
      f"{REFUSAL_TABLE[['system', 'correct_refusals']].to_dict('records')}",
      f"يجب أن يحوي جدول الرفض النظامين مع عدد الرفض الصحيح لكلٍّ — والناتج "
      f"{REFUSAL_TABLE[['system', 'correct_refusals']].to_dict('records')}")

V2_REFUSALS = int(REFUSAL_TABLE.loc[0, "correct_refusals"])
V4_REFUSALS = int(REFUSAL_TABLE.loc[1, "correct_refusals"])
SAME_RECALL = np.isclose(REFUSAL_TABLE.loc[0, "mean_recall_30"],
                         REFUSAL_TABLE.loc[1, "mean_recall_30"])
check(V4_REFUSALS > V2_REFUSALS and SAME_RECALL,
      f"V4 must refuse more out-of-scope questions than V2 ({V4_REFUSALS} against {V2_REFUSALS}) "
      f"while their scores on the answerable questions stay together (identical recall: "
      f"{SAME_RECALL}). Both halves matter: the second is what makes the first meaningful",
      f"يجب أن يرفض V4 من الأسئلة خارج النطاق أكثر من V2 ({V4_REFUSALS} مقابل {V2_REFUSALS}) مع "
      f"بقاء درجاتهما على الأسئلة القابلة للإجابة متقاربة (استدعاء متطابق: {SAME_RECALL}). "
      f"والنصفان يهمّان: فالثاني هو ما يجعل الأول ذا معنى")

check(IMPROVED >= 3 or len(broken) > 0,
      f"the re-chunking fix must either improve at least three of the five targeted questions "
      f"(it improved {IMPROVED}) or the notebook must record why it did not — it broke "
      f"{len(broken)} other questions, and the set-wide mean went "
      f"{RECALL.recall.mean():.3f} → {AFTER.mean():.3f}. A fix measured only on what it was "
      f"aimed at is not measured",
      f"يجب أن يحسّن إصلاح إعادة التقطيع ثلاثة من الخمسة المستهدفة على الأقل (حسّن {IMPROVED}) أو "
      f"أن يسجّل الدفتر لماذا لم يفعل — فقد كسر {len(broken)} أسئلة أخرى، وتحرّك متوسّط المجموعة "
      f"{RECALL.recall.mean():.3f} ← {AFTER.mean():.3f}. والإصلاح المقيس على ما استُهدف به وحده غير مقيس")

check(HYBRID.mean() >= AFTER.mean() - 1e-9,
      f"hybrid retrieval must not be worse than dense alone on mean context recall — got "
      f"{HYBRID.mean():.3f} against {AFTER.mean():.3f}. Fusing two rankings cannot lose "
      f"information unless the fusion is wrong",
      f"يجب ألّا يكون الاسترجاع الهجين أسوأ من الكثيف وحده في متوسّط الاستدعاء — والناتج "
      f"{HYBRID.mean():.3f} مقابل {AFTER.mean():.3f}. فدمج ترتيبين لا يفقد معلومة إلا إذا كان الدمج خطأ")

REQUIRED = ["recall", "faithfulness", "relevance", "refused_v4", "correct_refusal",
            "latency_s", "total_tokens"]
POPULATED = {column: int(EVAL[column].notna().sum()) for column in REQUIRED}
check(all(POPULATED[column] >= len(ANSWERABLE) for column in
          ["recall", "faithfulness", "relevance"])
      and all(POPULATED[column] == len(EVAL) for column in
              ["refused_v4", "correct_refusal", "latency_s", "total_tokens"]),
      f"rag_eval.parquet must be populated: the three metrics on all {len(ANSWERABLE)} answerable "
      f"questions and the per-question columns on all {len(EVAL)} — got {POPULATED}",
      f"يجب أن يكون `rag_eval.parquet` مملوءًا: المقاييس الثلاثة على الأسئلة {len(ANSWERABLE)} "
      f"القابلة للإجابة، وأعمدة كل سؤال على {len(EVAL)} — والناتج {POPULATED}")

report()

## What's next

**Tomorrow is the last taught lab of the bootcamp** — two recommenders on one dataset, honest
ranking metrics, and a popularity baseline that will probably beat both of them. The habit is the
same one this lab is built on: a number with nothing beside it is not a result.

**Capstone M3 is due today**, and it is graded on what you did in this notebook rather than in
Wednesday's: metrics computed per component, a judge you calibrated, a refusal rate quoted next to
the scores it could have been traded against, and a fix you measured on the whole set rather than on
the five questions you were staring at.

<div dir="rtl" align="right">

## ما التالي

**غدًا آخر معمل مُدرَّس في المعسكر** — نظاما توصية على بيانات واحدة، ومقاييس ترتيب أمينة، وخطّ أساس
شعبيّ سيتفوّق عليهما على الأرجح. والعادة هي نفسها التي بُني عليها هذا المعمل: الرقم بلا شيء بجواره
ليس نتيجة.

**ويُسلَّم إنجاز مشروع التخرّج الثالث اليوم**، وهو مُقيَّم على ما فعلته في هذا الدفتر لا في دفتر
الأربعاء: مقاييس محسوبة لكل مكوّن، وحَكَم عايرته، ومعدّل رفض مذكور بجوار الدرجات التي كان يمكن
مقايضته بها، وإصلاح قِسته على المجموعة كلها لا على الأسئلة الخمسة التي كنت تحدّق فيها.

</div>